# Week 3: class imbalance remedies
Author: Selina. Adapted from the original Drive fair-comparison notebook.

Compare full-image BCE, mixed foreground/random crops, focal loss (gamma 2), and weighted BCE (5, 10, 20, 40). CNN without skips, width 8, Adam 0.0003, seed 163. All arms use the original splits and train-only normalization.

The crop arm uses eight 1024×64 crops per parent, each foreground-biased with probability 0.33. This equals the full-image sampled-pixel budget; it does not guarantee a fixed foreground/random ratio.


In [ ]:
from cs163_rfi import paths
from cs163_rfi import imbalance
from IPython.display import display

print("Data:", paths.DATA_DIR)
print("Results:", imbalance.OUT)


## 1. Verify data and normalization


In [ ]:
catalog = imbalance.prepare()
display(catalog.groupby("split").size())
print("Normalization:", imbalance.MEAN, imbalance.STD)


## 2. Train candidates
The highest validation AP selects each checkpoint. Minimum 20 epochs, patience 10, improvement anchor 0.001; a 100-epoch budget that fails to plateau stops with an error. Checkpoints are external to git.


In [ ]:
validation_runs = imbalance.run_candidates()
display(validation_runs.sort_values("validation_ap", ascending=False))


## 3. Freeze choices before test evaluation
Choose the positive weight by validation AP. For each method, choose its threshold by validation F1, then freeze it for test evaluation. These operating points differ from the Week 2 clean-control FPR protocol. Test AP is not used to pick a method. The historical test set has already been examined.


In [ ]:
RUN_TEST = False  # Enable only after reviewing/fixing validation choices.
if RUN_TEST:
    display(imbalance.evaluate_selected(validation_runs))


## 4. Compare the saved reference
This reads historical results; it does not substitute for fresh inference.


In [ ]:
import pandas as pd
display(pd.read_csv(paths.RESULTS_DIR / "reference/imbalance_f1_comparison.csv"))


In [ ]:
print("Saved:", imbalance.OUT)
